# 🤖 Agente IA — Plan de Transmisión (RAG)

Este notebook crea un asistente que **responde preguntas sobre el informe del Plan de Transmisión** (PDF) usando la técnica **RAG** (*Retrieval-Augmented Generation*).

**Modelos gratuitos que usamos:**
- 🧠 **Groq** (`openai/gpt-oss-120b`) → redacta las respuestas. Gratis con una API key.
- 🔎 **HuggingFace (local)** → convierte el texto en vectores para poder buscarlo. Corre en tu PC, sin costo ni internet.

**Cómo funciona un RAG, en 4 pasos:**
1. Partimos el PDF en trozos pequeños (*chunks*).
2. Cada trozo se convierte en un vector (*embedding*) que captura su significado.
3. Cuando preguntas, buscamos los trozos más parecidos a tu pregunta.
4. Le pasamos esos trozos al modelo, que redacta la respuesta **solo con esa información**.

> Trabajamos con **UN solo PDF**: *Propuesta Definitiva de Actualización del Plan de Transmisión 2027-2036*.

## Paso 1 — Tu API key de Groq (gratis)

1. Entra a **https://console.groq.com** y crea una cuenta gratis.
2. Ve a **API Keys → Create API Key** y cópiala.
3. Pégala abajo, reemplazando el contenido entre comillas.

⚠️ **No compartas esta key ni este notebook con la key adentro.** Es como una contraseña.

In [ ]:
import os

# Reemplaza el texto entre comillas por tu API key de Groq:
os.environ["GROQ_API_KEY"] = "PEGA_TU_API_KEY_AQUI"

# Pequeña verificación:
if os.environ["GROQ_API_KEY"].startswith("gsk_"):
    print("✅ API key cargada.")
else:
    print("⚠️ Revisa tu API key: las de Groq empiezan con 'gsk_'.")

### (Opcional) Ver qué modelos tienes disponibles en Groq

Groq cambia sus modelos cada cierto tiempo. Si el modelo del Paso 2 diera error `model_not_found`, ejecuta esta celda para ver los nombres disponibles y elige uno de la lista.

In [ ]:
import os, requests

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
)
print("Modelos disponibles en tu cuenta:\n")
for m in sorted(x["id"] for x in r.json()["data"]):
    print(" -", m)

## Paso 2 — Configurar los modelos

Aquí le decimos a LlamaIndex qué modelos usar:
- **`llm`**: el modelo que redacta respuestas (Groq / `openai/gpt-oss-120b`).
- **`embed_model`**: el que crea los vectores (HuggingFace, local).

`Settings` es la configuración global: una vez puesta, todo el notebook la usa.

> La **primera vez**, la línea del embedding descargará el modelo (~unos cientos de MB). Es una sola vez; luego queda en caché.

In [ ]:
import os
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 🧠 Modelo que redacta respuestas (gratis, en la nube de Groq).
#    Alternativas más rápidas: "openai/gpt-oss-20b"  o  "qwen/qwen3.8-27b".
Settings.llm = Groq(model="openai/gpt-oss-120b")

# 🔎 Modelo de embeddings (gratis, corre en tu PC). Multilingüe → funciona en español.
#    Usamos la caché estándar de HuggingFace para evitar un problema de descarga
#    incompleta con la caché propia de LlamaIndex.
cache_hf = os.path.join(os.path.expanduser("~"), ".cache", "huggingface", "hub")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    cache_folder=cache_hf,
)

print("✅ Modelos configurados.")

## Paso 3 — Cargar el PDF (página por página)

Aquí leemos el PDF **página por página** con `pypdf` y creamos un documento por cada página.

> ⚠️ **¿Por qué no usamos el lector automático `SimpleDirectoryReader`?**
> En este PDF, ese lector duplicaba el texto (¡9.7 millones de caracteres en vez de 640 mil!) y perdía los números de página. Leyéndolo nosotros con `pypdf` obtenemos:
> - **Texto limpio** (15× más pequeño → indexación mucho más rápida).
> - **El número de página** guardado en cada documento → así podremos citar "esto salió de la página X".

La función `cargar_pdfs` recorre **todos** los PDF de la carpeta `data/`.

In [ ]:
from pypdf import PdfReader
from pathlib import Path
from llama_index.core import Document

def cargar_pdfs(carpeta="data"):
    """Lee todos los PDF de la carpeta, un documento por página, con su número de página."""
    documentos = []
    for pdf_path in sorted(Path(carpeta).glob("*.pdf")):
        lector = PdfReader(str(pdf_path))
        for num, pagina in enumerate(lector.pages, start=1):
            texto = pagina.extract_text() or ""
            if texto.strip():  # ignoramos páginas vacías
                documentos.append(
                    Document(
                        text=texto,
                        metadata={"file_name": pdf_path.name, "page_label": str(num)},
                    )
                )
    return documentos

documents = cargar_pdfs("data")

print(f"📄 Se cargaron {len(documents)} páginas.")
print(f"   Total de caracteres: {sum(len(d.text) for d in documents):,}")
print("\nMuestra del contenido de la primera página:\n")
print(documents[0].text[:500])

## Paso 4 — Crear el índice

`VectorStoreIndex.from_documents(...)` hace el trabajo pesado del RAG:
- parte los documentos en trozos,
- calcula el embedding (vector) de cada trozo,
- los guarda en memoria para poder buscarlos.

Luego `as_query_engine(similarity_top_k=5)` crea el "motor de preguntas". El parámetro **`similarity_top_k`** define **cuántos trozos** del informe se recuperan por pregunta (más trozos = más contexto = respuestas más completas, pero un poco más lento).

> Esto puede tardar un poco la primera vez (calcula vectores de todo el PDF).

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

# similarity_top_k = cuántos trozos del informe recupera por cada pregunta.
# Por defecto son 2. Lo subimos a 5 para que las respuestas usen MÁS contexto
# (mejor para preguntas cuyo dato está repartido, como los costos).
# Si quieres respuestas aún más completas, puedes probar 6-8 (más lento y usa más tokens).
query_engine = index.as_query_engine(similarity_top_k=5)

print("✅ Índice creado. ¡Ya podemos hacer preguntas! (usando 5 trozos de contexto)")

## Paso 5 — Hacer preguntas 🎯

Escribe tu pregunta y el asistente responderá **según el informe**. Cambia el texto de `pregunta` por lo que quieras.

In [ ]:
pregunta = "¿De qué trata el informe y qué periodo cubre el plan de transmisión?"

respuesta = query_engine.query(pregunta)
print(respuesta.response)

### ¿De dónde sacó la respuesta? (fuentes)

Una gran ventaja del RAG: podemos ver **de qué páginas** salió la respuesta. Esto da confianza y permite verificar.

In [ ]:
for i, nodo in enumerate(respuesta.source_nodes, 1):
    pagina = nodo.metadata.get("page_label", "?")
    print(f"Fuente {i} — página {pagina} (relevancia {nodo.score:.3f})")
    print(nodo.text[:200].strip(), "...\n")

## 💬 Modo chat — pregunta lo que quieras

Dos formas de usarlo:

- **`preguntar("tu pregunta")`** → una función cómoda que muestra la respuesta y las páginas de origen.
- **Chat continuo** → una celda que te va pidiendo preguntas una tras otra (escribe `salir` para terminar). Ideal para mostrarle el asistente a tu jefe.

In [ ]:
def preguntar(texto, mostrar_fuentes=True):
    """Hace una pregunta al asistente y muestra la respuesta + las páginas de origen."""
    r = query_engine.query(texto)
    print("🤖", r.response)
    if mostrar_fuentes:
        print("\n📄 Fuentes:")
        for i, n in enumerate(r.source_nodes, 1):
            print(f"   {i}. página {n.metadata.get('page_label','?')} (relevancia {n.score:.3f})")
    return r

# Ejemplo (cámbialo por lo que quieras):
_ = preguntar("¿Qué proyectos se proponen para reforzar la zona de Lima?")

In [ ]:
# 💬 Chat continuo: escribe preguntas y pulsa Enter. Escribe 'salir' para terminar.
print("Asistente del Plan de Transmisión. Escribe 'salir' para terminar.\n")
while True:
    q = input("Tu pregunta: ").strip()
    if q.lower() in ("salir", "exit", "quit", ""):
        print("👋 Chat terminado.")
        break
    print()
    preguntar(q)
    print("\n" + "-" * 60)

## Paso 6 — Guardar el índice (para no reprocesar)

Crear el índice cuesta tiempo. Lo guardamos en disco (`vector_store/`) para reutilizarlo después sin volver a procesar el PDF.

La celda final muestra cómo **cargarlo de nuevo** en una sesión futura (sin repetir los pasos 3 y 4).

In [ ]:
index.storage_context.persist("vector_store")
print("💾 Índice guardado en la carpeta 'vector_store/'.")

In [ ]:
# --- Cómo recargar el índice guardado en el futuro (NO ejecutar ahora) ---
# from llama_index.core import StorageContext, load_index_from_storage
#
# storage = StorageContext.from_defaults(persist_dir="vector_store")
# index = load_index_from_storage(storage)
# query_engine = index.as_query_engine(similarity_top_k=5)
# print(query_engine.query("tu pregunta aquí").response)